# Tennis Play vs Non-Play Classification - ResNet18 Training
Ye notebook Google Colab par run karne ke liye hai. Make sure aap **Runtime -> Change runtime type** mein jake **T4 GPU** select karein.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os
import copy
import time

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 1. Dataset Setup
Aapko apna dataset Colab mein upload karna hoga. Dataset ka structure aisa hona chahiye:
`dataset/train/Play/`, `dataset/train/Non-Play/`
`dataset/val/Play/`, `dataset/val/Non-Play/`

In [ ]:
# Unzip your dataset here (agar zip upload kiya hai toh)
# !unzip dataset.zip -d dataset/

In [ ]:
data_dir = 'dataset' # Folder jahan data rakha hai

# Data Augmentation and Normalization
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# PyTorch Datasets and DataLoaders
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
print(f"Classes: {class_names}")

### 2. Model Setup (ResNet-18)

In [ ]:
# Load pretrained ResNet-18
model = models.resnet18(weights='IMAGENET1K_V1')

# Epoch 1-5 ke liye parameters freeze karna (Fine-tuning phase)
for param in model.parameters():
    param.requires_grad = False

# Last layer ko replace karna apne 2 classes (Play/Non-Play) ke liye
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(device)
criterion = nn.CrossEntropyLoss()

### 3. Training Loop

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=20):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)
        
        # Phase 2: Epoch 5 ke baad poore model ko unfreeze kar dena (Full Training)
        if epoch == 5:
            print("Unfreezing all layers for full training...")
            for param in model.parameters():
                param.requires_grad = True
            # Chota learning rate set karte hain
            for param_group in optimizer.param_groups:
                param_group['lr'] = 1e-4

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
# Start training
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3) # Start with training only last layer
model = train_model(model, criterion, optimizer, num_epochs=20)

### 4. Save the Model Weights

In [ ]:
# Yeh wahi .pth file banayega jo humne project mein use ki hai
torch.save(model.state_dict(), 'play_vs_nonplay.pth')
print("Model weights saved successfully!")